In [1]:
# Load and save regional ozone mortality in single files

In [2]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [3]:
# Number of samples: full distribution or stats
n_samples = 300

In [11]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop - 1}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/region/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/"

for ens_num in ensemble_members:
    print(f"Processing ensemble number {ens_num:02d}")

    ds_years = []
    for year in years:
        if n_samples <= 500:
            files = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*_{year}.nc"
            description = ("Regional mortality (COPD) due to ozone "
                           " - scripts by A.F. Wells (2025)")
            out_file = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

        else:
            files = f"Regional_mortality_stats_{model}_{scenario}_{ens_num:02d}_*_{year}.nc"
            description = ("Regional mortality (COPD) due to ozone "
                           "statistics: including mean, median, "
                           "and the 95% CI - scripts by A.F. Wells (2025)")
            out_file = f"Regional_mortality_stats_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

        file_path = os.path.join(MORT_DIR, files)
        in_files = sorted(glob.glob(file_path))

        das = []
        for file in in_files:
            da = xr.open_dataarray(file)
            das.append(da)
        combined = xr.concat(das, "region")

        ds_years.append(combined)

    ds_mort = xr.concat(
        ds_years,
        dim=xr.DataArray(
            years,
            dims="year",
            name="year")
    )

    ds_mort.attrs["description"] = description
    ds_mort.attrs["model"] = model
    ds_mort.attrs["scenario"] = scenario
    ds_mort.attrs["ensemble_number"] = ens_num

    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    ds_mort.to_netcdf(out_path, engine="h5netcdf", encoding={"region": {"dtype": str}})

print("All processing complete.")

Processing ensemble number 01


ValueError: coordinate 'region' not present in all datasets.